# ViT Creation

7-class task (1 normal + 6 abnormal)

Like we said in the proposal, we follow Dosovitskiy et al., 2021's ViT implementation with 16x16 patches, linear patch embedding, positional embeddings + class token, Transformer encoder, 7-way head.

In [35]:
# ! pip3 install transformers
# ! pip3 install numpy
# ! pip3 install pandas
# ! pip3 install torch
# ! pip3 install torchvision
# ! pip3 install pillow
# ! pip3 install scikit-learn
# ! pip3 install matplotlib

## Configuration

In [36]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from transformers import ViTConfig, ViTForImageClassification
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import os

In [37]:
PRETRAINED = True # False = train from scratch
AUGMENT = True
EPOCHS = 10
BATCH_SIZE = 32
LR = 3e-5 if PRETRAINED else 3e-4
IMG_SIZE = 224
ROOT = "/Users/shreyanakum/Documents/CS184A/CS184A-Project"
DATA_CSV_LOC = ROOT + "/" + "DATA_PP"

device = "cpu" # Mac
print(device)

cpu


## Loading Data

In [38]:
train_df = pd.read_csv(DATA_CSV_LOC + "/train.csv")
val_df = pd.read_csv(DATA_CSV_LOC + "/val.csv")
test_df = pd.read_csv(DATA_CSV_LOC + "/test.csv")

classes = sorted(train_df["label"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
print(class_to_idx)
print(train_df["label"].value_counts())

{'ABE': 0, 'BLA': 1, 'FGC': 2, 'HAC': 3, 'KSC': 4, 'LYI': 5, 'normal': 6}
label
normal    94757
BLA        8381
HAC         286
LYI          45
FGC          33
KSC          30
ABE           6
Name: count, dtype: int64


In [39]:
normalize = transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    normalize,
])

aug_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
    transforms.ColorJitter(0.1, 0.1, 0.1),
    transforms.ToTensor(),
    normalize,
])

class CellDataset(Dataset):
    def __init__(self, df, transform):
        self.paths = [os.path.join(ROOT, p) for p in df["filepath"]]
        self.labels = [class_to_idx[l] for l in df["label"]]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), self.labels[i]

train_set = CellDataset(train_df, aug_transform if AUGMENT else eval_transform)
val_set = CellDataset(val_df, eval_transform)
test_set = CellDataset(test_df, eval_transform)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)

## Model Definition

In [40]:
if PRETRAINED:
    model = ViTForImageClassification.from_pretrained(
        "google/vit-base-patch16-224-in21k", num_labels=7)
else:
    config = ViTConfig(image_size=IMG_SIZE, patch_size=16, hidden_size=384,
                       num_hidden_layers=6, num_attention_heads=6,
                       intermediate_size=1536, num_labels=7)
    model = ViTForImageClassification(config)

model = model.to(device)

# class weights because of the class imbalance
counts = np.bincount(train_set.labels, minlength=7)
weights = torch.tensor(counts.sum() / (7 * counts), dtype=torch.float).to(device)

loss_fn = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 5648.43it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Training the Model

In [41]:
from tqdm.auto import tqdm

def evaluate(loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Validation", leave=False):
            x, y = x.to(device), y.to(device)

            out = model(pixel_values=x).logits
            total_loss += loss_fn(out, y).item() * len(y)
            all_preds += out.argmax(1).cpu().tolist()
            all_labels += y.cpu().tolist()

    return total_loss / len(all_labels), all_preds, all_labels


best_f1 = 0
train_losses, val_losses, val_f1s = [], [], []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    print(f"\nStarting epoch {epoch+1}/{EPOCHS}", flush=True)
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for x, y in progress:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(pixel_values=x).logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        progress.set_postfix(batch_loss=f"{loss.item()}")

    train_loss = total_loss / len(train_set)

    print("Training complete. Starting validation...", flush=True)
    val_loss, preds, labels = evaluate(val_loader)
    val_f1 = f1_score(labels, preds, average="macro")
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_f1s.append(val_f1)

    print(
        f"Epoch {epoch+1}: "
        f"train loss {train_loss}, "
        f"val loss {val_loss}, "
        f"val macro F1 {val_f1}",
        flush=True
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "best_vit.pt")
        print("Done", flush=True)


Starting epoch 1/10


Epoch 1:   3%|▎         | 106/3236 [06:26<3:10:08,  3.64s/it, batch_loss=0.5711090564727783]


KeyboardInterrupt: 

In [ ]:
plt.plot(train_losses, label="train loss")
plt.plot(val_losses, label="val loss")
plt.plot(val_f1s, label="val macro F1")
plt.xlabel("epoch")
plt.legend()
plt.show()

## Hyperparameter Tuning

In [ ]:
# TODO: @janprasad